# 01 - Unit Economics Model

This notebook rebuilds the first version of the Provider Waste-to-Fertilizer feasibility model in Python.

The goal is not to create an ML model yet. The goal is to make the calculation logic transparent, testable, and ready for future dashboard deployment.


## Modeling Questions

1. How much waste is processed each month?
2. How much fertilizer is produced?
3. What revenue comes from fertilizer sales and waste service fees?
4. What are the major operating costs?
5. What are the payback period, ROI, and IRR under CapEx?
6. Which assumptions should be validated next?


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.financial_model import FeasibilityInputs, choose_machine, load_machine_specs, summarize_feasibility
from src.scenario_analysis import load_scenarios, run_scenarios


## Load Extracted Assumptions


In [ ]:
machine_specs = load_machine_specs(ROOT / 'data' / 'processed' / 'machine_specs.csv')
waste_types = pd.read_csv(ROOT / 'data' / 'processed' / 'waste_type_assumptions.csv')
scenarios = load_scenarios(ROOT / 'data' / 'processed' / 'scenario_defaults.csv')

machine_specs.head()


## Base Case

The base case is intentionally simple. After the formula structure is reviewed, this should be reconciled with the original Excel workbook.


In [ ]:
base_inputs = FeasibilityInputs(
    business_model='capex',
    waste_tons_per_day=40,
    collection_rate=0.85,
    days_per_month=26,
    conversion_rate=0.70,
    fertilizer_price_usd_per_ton=250,
    service_fee_usd_per_ton=50,
    electricity_rate_usd_per_kwh=0.2033,
    enzyme_cost_usd_per_ton_waste=15,
    labor_cost_usd_per_month=10_000,
    maintenance_rate_annual=0.05,
    machine_discount_rate=0.20,
    grant_rate=0.10,
)

machine = choose_machine(machine_specs, base_inputs.waste_tons_per_day)
summary = summarize_feasibility(base_inputs, machine)
pd.Series(summary)


## Scenario Comparison


In [ ]:
scenario_results = run_scenarios(machine_specs, scenarios, business_model='capex')
scenario_results[[
    'scenario', 'machine_model', 'machine_count', 'monthly_waste_tons',
    'monthly_revenue', 'monthly_operating_cost', 'monthly_cash_flow',
    'initial_investment', 'payback_years', 'roi', 'irr'
]]


## Validation Notes

Next, compare this notebook against one known scenario from the source Excel model. Any mismatch should be explained as one of:

- different machine capacity interpretation,
- different revenue-sharing logic,
- missing cost item,
- different treatment of grants, loans, or down payments,
- cash flow vs accounting profit mismatch.
